In [ ]:
# Cell 1 - run information (required by the brief)
import importlib.metadata as _meta, subprocess, sys, random, numpy as np, torch

SEED = 266
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # our GPU has 24 GB but the brief says everything must fit on 16 GB, so we hold ourselves to that
    _total = torch.cuda.get_device_properties(0).total_memory
    torch.cuda.set_per_process_memory_fraction(min(1.0, 16e9 / _total), 0)

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
BUILT_FROM = "808fa63"  # the repo commit this notebook was generated from


def _version(name):
    try:
        return _meta.version(name)
    except _meta.PackageNotFoundError:
        return "not installed yet"


print({"commit": commit or f"no git here, notebook built from {BUILT_FROM}", "built_from": BUILT_FROM, "python": sys.version.split()[0],
       "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})
print({name: _version(name) for name in ['diffusers', 'transformers', 'peft', 'accelerate', 'datasets', 'lpips', 'torchmetrics']})

# DATA 266 Lab 2, Part 2 baseline (Kavan)

Muhammed leads Part 2 in `p2_finetune.ipynb`. The brief says the partner who is not leading still runs the whole pipeline at least as a baseline, so this is that baseline: clean the data, train one LoRA, generate, evaluate.

It also gives Part 3 a working LoRA until Muhammed's final model is ready: at the end we upload it to Hugging Face (`Kavan1804/data266-p2-baseline-kavan-lora`), and `p3_agents.ipynb` downloads it from there. **Numbers here are not the Part 2 results for the report.** Those come from `p2_finetune.ipynb`.

Everything this notebook needs downloads by itself. For the upload, add a Hugging Face write token as `HF_TOKEN`: a Colab secret (key icon), an environment variable, or `huggingface-cli login` once on this machine. Never paste it into the notebook.

In [ ]:
%pip install -q diffusers==0.33.1 transformers==4.51.3 peft==0.15.2 accelerate==1.6.0 datasets==3.5.1 lpips==0.1.4 torchmetrics==1.7.1 ftfy==6.3.1 tensorboard pandas pillow matplotlib

In [ ]:
# Where are we running? Paths are relative, so the same notebook works on Colab, Kaggle or a local Jupyter.
import importlib.util, os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = importlib.util.find_spec("google.colab") is not None
here = Path.cwd().resolve()
# if the notebook happens to sit inside our repo, deliverables go straight into it
REPO = next((p for p in [here, *here.parents] if all((p / d).is_dir() for d in ("eval", "src", "notebooks"))), None)
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB and Path("/content").is_dir() else (REPO or here)
# deliverables use the repo's folder layout, so the final ZIP unzips straight into the repo
OUT_ROOT = REPO or WORK / "lab2_repo_files"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("environment:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local jupyter")
print("repo found:", REPO, "| working folder:", WORK, "| deliverables folder:", OUT_ROOT)
print("cuda:", torch.cuda.is_available())


def hf_token():
    """Hugging Face token from the HF_TOKEN variable, Colab or Kaggle secrets, or an earlier `huggingface-cli login`.
    It is never typed into the notebook."""
    token = os.environ.get("HF_TOKEN")
    if not token and ON_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token and ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        try:
            from huggingface_hub import get_token
            token = get_token()
        except Exception:
            token = None
    if token:
        os.environ["HF_TOKEN"] = token  # diffusers and huggingface_hub read it from here
    return token

## Clean and split the data
We drop images that do not decode and exact duplicates, then split 90/10 with seed 266. Test images are never used for training.

In [ ]:
from datasets import Image as DatasetImage, load_dataset


def load_naruto():
    # the brief's lambdalabs/ name is now a redirect to lambda/, so we try both
    for name in ("lambda/naruto-blip-captions", "lambdalabs/naruto-blip-captions"):
        try:
            return name, load_dataset(name, split="train")
        except Exception as err:
            print("could not load", name, "->", type(err).__name__, str(err)[:150])
    raise RuntimeError("The Naruto dataset could not be loaded.")


import hashlib, io, json, time
from PIL import Image

OUT = WORK / "outputs" / "p2_baseline_kavan"
TRAIN_DIR, TEST_DIR = OUT / "data" / "train", OUT / "data" / "test"
OUT.mkdir(parents=True, exist_ok=True)
name, raw = load_naruto()
raw = raw.cast_column("image", DatasetImage(decode=False))
seen, keep, corrupted, duplicates = set(), [], 0, 0
for i, row in enumerate(raw):
    data = row["image"]["bytes"] or open(row["image"]["path"], "rb").read()
    try:
        Image.open(io.BytesIO(data)).convert("RGB")
    except Exception:
        corrupted += 1
        continue
    digest = hashlib.md5(data).hexdigest()
    if digest in seen:
        duplicates += 1
        continue
    seen.add(digest)
    keep.append(i)
random.Random(SEED).shuffle(keep)
cut = int(len(keep) * 0.9)
train_ids, test_ids = keep[:cut], keep[cut:]
print({"dataset": name, "rows": len(raw), "corrupted": corrupted, "duplicates": duplicates, "train": len(train_ids), "test": len(test_ids)})
for folder, ids in ((TRAIN_DIR, train_ids), (TEST_DIR, test_ids)):
    folder.mkdir(parents=True, exist_ok=True)
    with open(folder / "metadata.jsonl", "w") as f:
        for i in ids:
            row = raw[int(i)]
            data = row["image"]["bytes"] or open(row["image"]["path"], "rb").read()
            path = folder / f"{i:05d}.png"
            if not path.exists():
                Image.open(io.BytesIO(data)).convert("RGB").resize((512, 512), Image.LANCZOS).save(path)
            f.write(json.dumps({"file_name": path.name, "text": row["text"]}) + "\n")
print("captions look like:", json.loads((TEST_DIR / "metadata.jsonl").read_text().splitlines()[0])["text"])

## Train one LoRA
We use the diffusers example script from the same version we pin (0.33.1), at rank 4. A subprocess does not inherit our memory limit, so a small launcher applies the 16 GB cap inside it and prints peak memory at the end.

If weights and a training log already exist, we reuse them and say so. Delete `outputs/p2_baseline_kavan/lora` to train again.

In [ ]:
import urllib.request

MAX_STEPS, RANK, LR, BATCH = 1500, 4, 1e-4, 1
LORA_DIR, META = OUT / "lora", OUT / "run_meta.json"
SCRIPT = OUT / "train_text_to_image_lora.py"
URL = "https://raw.githubusercontent.com/huggingface/diffusers/v0.33.1/examples/text_to_image/train_text_to_image_lora.py"
LAUNCHER = OUT / "capped_launcher.py"
LAUNCHER.write_text("""
import atexit, runpy, sys, torch
if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(min(1.0, 16e9 / torch.cuda.get_device_properties(0).total_memory), 0)
    atexit.register(lambda: print("PEAK_GPU_GB", round(torch.cuda.max_memory_allocated() / 1e9, 3), flush=True))
script = sys.argv[1]
sys.argv = sys.argv[1:]
runpy.run_path(script, run_name="__main__")
""")
if (LORA_DIR / "pytorch_lora_weights.safetensors").exists() and META.exists():
    run_meta = json.loads(META.read_text())
    print("REUSING an earlier training run, not training again:")
else:
    if not SCRIPT.exists():
        urllib.request.urlretrieve(URL, SCRIPT)
    command = [sys.executable, str(LAUNCHER), str(SCRIPT), "--pretrained_model_name_or_path", "stable-diffusion-v1-5/stable-diffusion-v1-5",
               "--train_data_dir", str(TRAIN_DIR), "--caption_column", "text", "--resolution", "512", "--random_flip",
               "--train_batch_size", str(BATCH), "--max_train_steps", str(MAX_STEPS), "--learning_rate", str(LR), "--lr_scheduler", "cosine",
               "--lr_warmup_steps", "0", "--rank", str(RANK), "--seed", str(SEED), "--mixed_precision", "fp16", "--output_dir", str(LORA_DIR),
               "--checkpointing_steps", "500", "--checkpoints_total_limit", "1", "--dataloader_num_workers", "2", "--report_to", "tensorboard"]
    start = time.perf_counter()
    peak = None
    with open(OUT / "train_log.txt", "w") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            if line.startswith("PEAK_GPU_GB"):
                peak = float(line.split()[1])
            if "Steps:" not in line:
                print(line, end="")
        code_ = process.wait()
    assert code_ == 0, f"training failed with exit code {code_}, see {OUT / 'train_log.txt'}"
    seconds = time.perf_counter() - start
    run_meta = {"rank": RANK, "learning_rate": LR, "steps": MAX_STEPS, "batch_size": BATCH, "seed": SEED, "train_seconds": seconds,
                "gpu_hours": seconds / 3600, "peak_gpu_gb": peak, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
    META.write_text(json.dumps(run_meta, indent=2))
if "trainable_params" not in run_meta:
    # LoRA only trains the adapter weights, so counting the saved tensors gives the trainable parameters
    from safetensors.torch import load_file
    run_meta["trainable_params"] = int(sum(t.numel() for t in load_file(str(LORA_DIR / "pytorch_lora_weights.safetensors")).values()))
    META.write_text(json.dumps(run_meta, indent=2))
print(run_meta)

## Generate
Same sampler settings for both models. We use 4 test captions plus 2 out-of-domain prompts, with seeds 0 and 1. The SD safety checker stays on, and we count what it flags.

In [ ]:
import gc
from diffusers import StableDiffusionPipeline

captions = [json.loads(x)["text"] for x in (TEST_DIR / "metadata.jsonl").read_text().splitlines()][:4]
PROMPTS = captions + ["a photo of a red bicycle next to a lake", "a bowl of fruit on a wooden table"]
SAMPLER = dict(num_inference_steps=30, guidance_scale=7.5, height=512, width=512)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
GEN = OUT / "generated"
records = []
for model_name in ("base", "lora"):
    pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=dtype).to(device)
    pipe.set_progress_bar_config(disable=True)
    if model_name == "lora":
        pipe.load_lora_weights(str(LORA_DIR), weight_name="pytorch_lora_weights.safetensors")
    (GEN / model_name).mkdir(parents=True, exist_ok=True)
    for p, prompt in enumerate(PROMPTS):
        for seed in (0, 1):
            out = pipe(prompt, generator=torch.Generator(device).manual_seed(seed), **SAMPLER)
            path = GEN / model_name / f"p{p}_s{seed}.png"
            out.images[0].save(path)
            records.append({"model": model_name, "prompt_id": p, "prompt": prompt, "seed": seed, "path": str(path), "nsfw": bool(out.nsfw_content_detected and out.nsfw_content_detected[0])})
    del pipe
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("images:", len(records), "| flagged by the safety checker:", sum(r["nsfw"] for r in records))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(PROMPTS), 2, figsize=(7, 3.4 * len(PROMPTS)))
for p, prompt in enumerate(PROMPTS):
    for c, model_name in enumerate(("base", "lora")):
        axes[p, c].imshow(Image.open(GEN / model_name / f"p{p}_s0.png"))
        axes[p, c].set_title(f"{model_name}: {prompt[:38]}", fontsize=8)
        axes[p, c].axis("off")
plt.tight_layout()
(OUT_ROOT / "reports" / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_ROOT / "reports" / "figures" / "p2_baseline_kavan_grid.png", dpi=110, bbox_inches="tight")
plt.show()

## Evaluate
CLIP score (prompt vs image), LPIPS between the two seeds of each prompt, and KID against test images. With this few images KID is very noisy, so it is only a sanity check.

In [ ]:
import lpips
import torchvision.transforms.functional as TF
from torchmetrics.image.kid import KernelInceptionDistance
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
loss_fn = lpips.LPIPS(net="alex", verbose=False).to(device)


def clip_score(path, text):
    inputs = clip_processor(text=[text], images=Image.open(path).convert("RGB"), return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.inference_mode():
        out = clip(**inputs)
    a, b = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True), out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return float((a * b).sum())


def as_tensor(path):
    return TF.to_tensor(Image.open(path).convert("RGB")).unsqueeze(0).to(device) * 2 - 1


results = {"note": "Kavan's baseline, not the Part 2 results", **run_meta}
in_domain = [r for r in records if r["prompt_id"] < 4]
real = [TF.pil_to_tensor(Image.open(p).convert("RGB")) for p in sorted(TEST_DIR.glob("*.png"))]
for model_name in ("base", "lora"):
    subset = [r for r in records if r["model"] == model_name]
    scores = [clip_score(r["path"], r["prompt"]) for r in subset]
    with torch.inference_mode():
        diversity = [float(loss_fn(as_tensor(GEN / model_name / f"p{p}_s0.png"), as_tensor(GEN / model_name / f"p{p}_s1.png"))) for p in range(len(PROMPTS))]
    fake = [TF.pil_to_tensor(Image.open(r["path"]).convert("RGB")) for r in in_domain if r["model"] == model_name]
    kid = KernelInceptionDistance(subset_size=min(len(fake), len(real)) - 1, subsets=20, normalize=False).to(device)
    kid.update(torch.stack(real).to(device), real=True)
    kid.update(torch.stack(fake).to(device), real=False)
    kid_mean, kid_std = kid.compute()
    results[model_name] = {"clip_score": sum(scores) / len(scores), "lpips_diversity": sum(diversity) / len(diversity), "kid_mean": float(kid_mean), "kid_std": float(kid_std)}
print(json.dumps(results, indent=2))
results_dir = OUT_ROOT / "reports" / "results"
results_dir.mkdir(parents=True, exist_ok=True)
(results_dir / "p2_baseline_kavan.json").write_text(json.dumps(results, indent=2))
(OUT / "generated" / "records.json").write_text(json.dumps(records, indent=2))

## Upload the LoRA to Hugging Face
The brief wants weights on Hugging Face, not GitHub, and Part 3 loads this LoRA from there. The repo is public so Part 3 and the graders can read it. If no token is found we say so and carry on, because the ZIP below still holds the weights.

In [ ]:
HF_LORA_REPO = os.environ.get("HF_LORA_REPO", "Kavan1804/data266-p2-baseline-kavan-lora")
HF_PRIVATE = False
card = f"""---
base_model: stable-diffusion-v1-5/stable-diffusion-v1-5
library_name: diffusers
tags: [lora, text-to-image, stable-diffusion]
license: other
---
# DATA 266 Lab 2, Part 2 baseline LoRA (Kavan, Team 24)

Partner baseline for our class lab, not our final Part 2 model.

- **Base model:** stable-diffusion-v1-5/stable-diffusion-v1-5 at 512x512
- **Data:** {name}, cleaned and split 90/10 with seed {SEED} ({len(train_ids)} training images)
- **Training:** LoRA rank {run_meta['rank']}, learning rate {run_meta['learning_rate']}, {run_meta['steps']} steps, batch size {run_meta['batch_size']}, seed {run_meta['seed']}, fp16
- **Cost:** {run_meta['gpu_hours']:.2f} GPU-hours on {run_meta['gpu']}, peak memory {run_meta['peak_gpu_gb']} GB
- **Results (small sanity check, see the notebook):** CLIP score {results['lora']['clip_score']:.3f} vs base {results['base']['clip_score']:.3f}

**License and limits:** Naruto characters are copyrighted and the dataset page lists no license, so this LoRA is for non-commercial research only. It was trained on a small anime dataset and copies that style even when you do not want it.
"""
upload = {"hf_repo": HF_LORA_REPO, "uploaded": False}
token = hf_token()
if not token:
    print("NOT uploaded: no Hugging Face token found. Part 3 can still use the local copy if it runs in this same folder.")
else:
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        api.create_repo(HF_LORA_REPO, repo_type="model", private=HF_PRIVATE, exist_ok=True)
        (OUT / "README.md").write_text(card)
        for local, remote in ((LORA_DIR / "pytorch_lora_weights.safetensors", "pytorch_lora_weights.safetensors"), (META, "run_meta.json"), (OUT / "README.md", "README.md")):
            api.upload_file(path_or_fileobj=str(local), path_in_repo=remote, repo_id=HF_LORA_REPO, repo_type="model")
        upload["uploaded"] = True
        print("uploaded to https://huggingface.co/" + HF_LORA_REPO)
    except Exception as err:
        print("UPLOAD FAILED:", type(err).__name__, err)
(OUT_ROOT / "reports" / "results" / "p2_baseline_kavan_upload.json").write_text(json.dumps(upload, indent=2))

## What we observed
Fill this in after the run, using only what the outputs above show: did the LoRA move in-domain prompts toward the style, and did it hurt out-of-domain prompts? We leave it open until there are real outputs.

In [ ]:
# Last cell - put this notebook's deliverables in one ZIP. It uses the repo's folder layout, so unzipping it
# at the repo root puts every file where it belongs. Original project files are only read, never changed.
import json as _json, shutil, zipfile

KEEP = ['outputs/p2_baseline_kavan/lora/pytorch_lora_weights.safetensors', 'outputs/p2_baseline_kavan/run_meta.json', 'outputs/p2_baseline_kavan/train_log.txt', 'outputs/p2_baseline_kavan/generated', 'reports/results/p2_baseline_kavan.json', 'reports/results/p2_baseline_kavan_upload.json', 'reports/figures/p2_baseline_kavan_grid.png']
export_dir = WORK / "export_p2_baseline_kavan"
if export_dir.exists():
    shutil.rmtree(export_dir)
(export_dir / "notebooks").mkdir(parents=True)

# 1) the executed notebook. On Colab the .ipynb is not on disk, so we ask the browser for it.
nb_target, saved = export_dir / "notebooks" / "p2_baseline_kavan.ipynb", False
if ON_COLAB:
    try:
        from google.colab import _message
        nb_target.write_text(_json.dumps(_message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]))
        saved = True
    except Exception as err:
        print("could not fetch the notebook from Colab:", type(err).__name__, err)
if not saved:
    for candidate in [Path("/kaggle/src/script.ipynb"), here / "p2_baseline_kavan.ipynb", *([REPO / "notebooks" / "p2_baseline_kavan.ipynb"] if REPO else [])]:
        if candidate.exists():
            shutil.copy(candidate, nb_target)
            saved = True
            break
if saved:
    print("notebook included. It has every output except this last cell, so also do File > Download .ipynb once this finishes.")
else:
    print("WARNING: notebook file not found. Do File > Download .ipynb and put it in notebooks/p2_baseline_kavan.ipynb in the repo.")

# 2) the deliverables, small files only (no raw data, model caches or secrets)
skip_parts = {"__pycache__", ".git", "chroma", "hf_cache", "data", "checkpoints"}
for rel in KEEP:
    source = OUT_ROOT / rel if (OUT_ROOT / rel).exists() else WORK / rel
    if source.is_file():
        (export_dir / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(source, export_dir / rel)
    elif source.is_dir():
        for path in source.rglob("*"):
            if path.is_file() and not (skip_parts & set(path.relative_to(source).parts)) and path.stat().st_size < 300e6:
                target = export_dir / rel / path.relative_to(source)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(path, target)
    else:
        print("not found, skipped:", rel)

zip_path = WORK / "DATA266_Lab2_p2_baseline_kavan.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(export_dir.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(export_dir))

print(f"Created ZIP: {zip_path.resolve()} ({zip_path.stat().st_size / 1e6:.1f} MB). Unzip it at the repo root.")
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        print(f"  {info.file_size / 1e6:8.2f} MB  {info.filename}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(zip_path.relative_to(Path.cwd())) if zip_path.is_relative_to(Path.cwd()) else str(zip_path)))
    except Exception:
        pass
    print("If no download started, take the ZIP from the path above (file browser or output panel).")